<a target="_blank" href="https://colab.research.google.com/github/eeg2025/startkit/blob/main/challenge_1.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Challenge 1: Cross-Task Transfer Learning!

## How can we use the knowledge from one EEG Decoding task into another?

Transfer learning is a widespread technique used in deep learning. It uses knowledge learned from one source task/domain in another target task/domain. It has been studied in depth in computer vision, natural language processing, and speech, but what about EEG brain decoding?

The cross-task transfer learning scenario in EEG decoding is remarkably underexplored in comparison to the developers of new models, [Aristimunha et al., (2023)](https://arxiv.org/abs/2308.02408), even though it can be much more useful for real applications, see [Wimpff et al. (2025)](https://arxiv.org/abs/2502.06828), [Wu et al. (2025)](https://arxiv.org/abs/2507.09882).

Our Challenge 1 addresses a key goal in neurotechnology: decoding cognitive function from EEG using the pre-trained knowledge from another. In other words, developing models that can effectively transfer/adapt/adjust/fine-tune knowledge from passive EEG tasks to active tasks.

The ability to generalize and transfer is something critical that we believe should be focused. To go beyond just comparing metrics numbers that are often not comparable, given the specificities of EEG, such as pre-processing, inter-subject variability, and many other unique components of this type of data.

This means your submitted model might be trained on a subset of tasks and fine-tuned on data from another condition, evaluating its capacity to generalize with task-specific fine-tuning.

__________

Note: For simplicity purposes, we will only show how to do the decoding directly in our target task, and it is up to the teams to think about how to use the passive task to perform the pre-training.

⚠️ **In case of colab, before starting, make sure you're on a GPU instance for faster training!** ⚠️

> If running on Google Colab, please request a GPU runtime by clicking `Runtime/Change runtime type` in the top bar menu, then selecting 'T4 GPU' under 'Hardware accelerator'.

In [44]:
# Identify whether a CUDA-enabled GPU is available
from pathlib import Path
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    msg ='CUDA-enabled GPU found. Training should be faster.'
else:
    msg = (
        "No GPU found. Training will be carried out on CPU, which might be "
        "slower.\n\nIf running on Google Colab, you can request a GPU runtime by"
        " clicking\n`Runtime/Change runtime type` in the top bar menu, then "
        "selecting \'T4 GPU\'\nunder \'Hardware accelerator\'."
    )
print(msg)

No GPU found. Training will be carried out on CPU, which might be slower.

If running on Google Colab, you can request a GPU runtime by clicking
`Runtime/Change runtime type` in the top bar menu, then selecting 'T4 GPU'
under 'Hardware accelerator'.


For the challenge, we will need two significant dependencies: `braindecode` and `eegdash`. The libraries will install PyTorch, Pytorch Audio, Scikit-learn, MNE, MNE-BIDS, and many other packages necessary for the many functions.

In [45]:
#@title ▶️ Install additional required packages for colab
# %pip install braindecode
# %pip install eegdash

In [46]:
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

from eegdash import EEGChallengeDataset

dataset_ccd = []

for i in range(1,12):
    if i == 5:
        continue
    # print (f"R{i}")
    # print(EEGChallengeDataset(task="contrastChangeDetection", release=f"R{i}", cache_dir=DATA_DIR, mini=True))
    dataset_ccd.append(EEGChallengeDataset(task="contrastChangeDetection",
                                      release=f"R{i}", cache_dir=DATA_DIR,
                                      mini=True))

print(len(dataset_ccd)) # should be 10

test_ccd = EEGChallengeDataset(task="contrastChangeDetection",
                                  release="R5", cache_dir=DATA_DIR,
                                  mini=True)



10


If you want to load the whole release you need change the `mini=False`.

In [47]:
# For visualization purposes, we will see just one object.

raw = dataset_ccd[0].datasets[0].raw  # get the Raw object of the first recording

In [48]:
# fig = raw.plot()

In [49]:
print(raw.info)
print(len(raw.info["ch_names"]))

# line does not work?
# fig = raw.plot_sensors() 

<Info | 9 non-empty values
 bads: []
 ch_names: E1, E2, E3, E4, E5, E6, E7, E8, E9, E10, E11, E12, E13, E14, ...
 chs: 129 EEG
 custom_ref_applied: False
 highpass: 0.0 Hz
 line_freq: 60.0
 lowpass: 50.0 Hz
 meas_date: 2025-08-19 00:00:59 UTC
 nchan: 129
 projs: []
 sfreq: 100.0 Hz
 subject_info: <subject_info | his_id: sub-NDARAC904DMU, sex: 2, birthday: 2014-04-17>
>
129


As you just realized, the eeg dash dataset object will download the dataset only when necessary, and in this case, only when we want to consume the raw data. To download all data directly, we recommend downloading the versions with Amazon API, or doing something like:

In [50]:
# from joblib import Parallel, delayed

# train_raws = Parallel(n_jobs=-1)(
#     delayed(lambda d: d.raw)(d) 
#     for subject in dataset_ccd
#     for d in subject.datasets
# )

# valid_raws = Parallel(n_jobs=-1)(
#     delayed(lambda d: d.raw)(d) for d in test_ccd.datasets
# )

In [69]:
#@title ▶️ Run this first to get all the utils functions for the epoching
import numpy as np
import pandas as pd
import mne
from mne_bids import get_bids_path_from_fname
from braindecode.datasets import BaseConcatDataset, WindowsDataset

# extract evets:
def build_trial_table(events_df: pd.DataFrame) -> pd.DataFrame:
    """One row per contrast trial with stimulus/response metrics."""
    events_df = events_df.copy()
    events_df["onset"] = pd.to_numeric(events_df["onset"], errors="raise")
    events_df = events_df.sort_values("onset", kind="mergesort").reset_index(drop=True)

    trials = events_df[events_df["value"].eq("contrastTrial_start")].copy()
    stimuli = events_df[events_df["value"].isin(["left_target", "right_target"])].copy()
    responses = events_df[events_df["value"].isin(["left_buttonPress", "right_buttonPress"])].copy()

    trials = trials.reset_index(drop=True)
    trials["next_onset"] = trials["onset"].shift(-1)
    trials = trials.dropna(subset=["next_onset"]).reset_index(drop=True)

    rows = []
    for _, tr in trials.iterrows():
        start = float(tr["onset"])
        end   = float(tr["next_onset"])

        stim_block = stimuli[(stimuli["onset"] >= start) & (stimuli["onset"] < end)]
        stim_onset = np.nan if stim_block.empty else float(stim_block.iloc[0]["onset"])

        if not np.isnan(stim_onset):
            resp_block = responses[(responses["onset"] >= stim_onset) & (responses["onset"] < end)]
        else:
            resp_block = responses[(responses["onset"] >= start) & (responses["onset"] < end)]

        if resp_block.empty:
            resp_onset = np.nan
            resp_type  = None
            feedback   = None
        else:
            resp_onset = float(resp_block.iloc[0]["onset"])
            resp_type  = resp_block.iloc[0]["value"]
            feedback   = resp_block.iloc[0]["feedback"]

        rt_from_stim  = (resp_onset - stim_onset) if (not np.isnan(stim_onset) and not np.isnan(resp_onset)) else np.nan
        rt_from_trial = (resp_onset - start)       if not np.isnan(resp_onset) else np.nan

        correct = None
        if isinstance(feedback, str):
            if feedback == "smiley_face": correct = True
            elif feedback == "sad_face":  correct = False

        rows.append({
            "trial_start_onset": start,
            "trial_stop_onset": end,
            "stimulus_onset": stim_onset,
            "response_onset": resp_onset,
            "rt_from_stimulus": rt_from_stim,
            "rt_from_trialstart": rt_from_trial,
            "response_type": resp_type,
            "correct": correct,
        })

    return pd.DataFrame(rows)

# Aux functions to inject the annot
def _to_float_or_none(x):
    return None if pd.isna(x) else float(x)

def _to_int_or_none(x):
    if pd.isna(x):
        return None
    if isinstance(x, (bool, np.bool_)):
        return int(bool(x))
    if isinstance(x, (int, np.integer)):
        return int(x)
    try:
        return int(x)
    except Exception:
        return None

def _to_str_or_none(x):
    return None if (x is None or (isinstance(x, float) and np.isnan(x))) else str(x)

def annotate_trials_with_target(raw, target_field="rt_from_stimulus", epoch_length=2.0,
                                require_stimulus=True, require_response=True):
    """Create 'contrast_trial_start' annotations with float target in extras."""
    fnames = raw.filenames
    assert len(fnames) == 1, "Expected a single filename"
    bids_path = get_bids_path_from_fname(fnames[0])
    events_file = bids_path.update(suffix="events", extension=".tsv").fpath

    events_df = (pd.read_csv(events_file, sep="\t")
                   .assign(onset=lambda d: pd.to_numeric(d["onset"], errors="raise"))
                   .sort_values("onset", kind="mergesort").reset_index(drop=True))

    trials = build_trial_table(events_df)

    if require_stimulus:
        trials = trials[trials["stimulus_onset"].notna()].copy()
    if require_response:
        trials = trials[trials["response_onset"].notna()].copy()

    if target_field not in trials.columns:
        raise KeyError(f"{target_field} not in computed trial table.")
    targets = trials[target_field].astype(float)

    onsets     = trials["trial_start_onset"].to_numpy(float)
    durations  = np.full(len(trials), float(epoch_length), dtype=float)
    descs      = ["contrast_trial_start"] * len(trials)

    extras = []
    for i, v in enumerate(targets):
        row = trials.iloc[i]

        extras.append({
            "target": _to_float_or_none(v),
            "rt_from_stimulus": _to_float_or_none(row["rt_from_stimulus"]),
            "rt_from_trialstart": _to_float_or_none(row["rt_from_trialstart"]),
            "stimulus_onset": _to_float_or_none(row["stimulus_onset"]),
            "response_onset": _to_float_or_none(row["response_onset"]),
            "correct": _to_int_or_none(row["correct"]),
            "response_type": _to_str_or_none(row["response_type"]),
        })

    new_ann = mne.Annotations(onset=onsets, duration=durations, description=descs,
                              orig_time=raw.info["meas_date"], extras=extras)
    raw.set_annotations(new_ann, verbose=False)
    return raw

# %% ------------------------------------ Add stimulus/response anchor events
def add_aux_anchors(raw, stim_desc="stimulus_anchor", resp_desc="response_anchor"):
    ann = raw.annotations
    mask = (ann.description == "contrast_trial_start")
    if not np.any(mask):
        return raw

    stim_onsets, resp_onsets = [], []
    stim_extras, resp_extras = [], []

    for idx in np.where(mask)[0]:
        ex = ann.extras[idx] if ann.extras is not None else {}
        t0 = float(ann.onset[idx])

        stim_t = ex["stimulus_onset"]
        resp_t = ex["response_onset"]

        if stim_t is None or (isinstance(stim_t, float) and np.isnan(stim_t)):
            rtt = ex["rt_from_trialstart"]
            rts = ex["rt_from_stimulus"]
            if rtt is not None and rts is not None:
                stim_t = t0 + float(rtt) - float(rts)

        if resp_t is None or (isinstance(resp_t, float) and np.isnan(resp_t)):
            rtt = ex["rt_from_trialstart"]
            if rtt is not None:
                resp_t = t0 + float(rtt)

        if (stim_t is not None) and not (isinstance(stim_t, float) and np.isnan(stim_t)):
            stim_onsets.append(float(stim_t))
            stim_extras.append(dict(ex, anchor="stimulus"))
        if (resp_t is not None) and not (isinstance(resp_t, float) and np.isnan(resp_t)):
            resp_onsets.append(float(resp_t))
            resp_extras.append(dict(ex, anchor="response"))

    new_onsets = np.array(stim_onsets + resp_onsets, dtype=float)
    if len(new_onsets):
        aux = mne.Annotations(
            onset=new_onsets,
            duration=np.zeros_like(new_onsets, dtype=float),
            description=[stim_desc]*len(stim_onsets) + [resp_desc]*len(resp_onsets),
            orig_time=raw.info["meas_date"],
            extras=stim_extras + resp_extras,
        )
        raw.set_annotations(ann + aux, verbose=False)
    return raw


def add_extras_columns(
    windows_concat_ds,
    original_concat_ds,
    desc="contrast_trial_start",
    keys=("target","rt_from_stimulus","rt_from_trialstart",
          "stimulus_onset","response_onset","correct","response_type"),
):
    float_cols = {"target","rt_from_stimulus","rt_from_trialstart",
                  "stimulus_onset","response_onset"}

    for win_ds, base_ds in zip(windows_concat_ds.datasets, original_concat_ds.datasets):
        ann = base_ds.raw.annotations
        idx = np.where(ann.description == desc)[0]
        if idx.size == 0:
            continue

        # Build per-trial dictionary from annotations
        per_trial_full = [
            {k: (ann.extras[i][k] if ann.extras is not None and k in ann.extras[i] else None)
             for k in keys}
            for i in idx
        ]

        md = win_ds.metadata.copy()
        first = (md["i_window_in_trial"].to_numpy() == 0)
        trial_ids = first.cumsum() - 1
        n_trials = trial_ids.max() + 1 if len(trial_ids) else 0

        # OPTION 3: Only keep per_trial entries for trials that actually have windows
        per_trial = per_trial_full[:n_trials]

        # Now lengths match
        for k in keys:
            vals = [per_trial[t][k] if t < len(per_trial) else None for t in trial_ids]
            if k == "correct":
                ser = pd.Series([None if v is None else int(bool(v)) for v in vals],
                                index=md.index, dtype="Int64")
            elif k in float_cols:
                ser = pd.Series([np.nan if v is None else float(v) for v in vals],
                                index=md.index, dtype="Float64")
            else:  # response_type
                ser = pd.Series(vals, index=md.index, dtype="string")

            md[k] = ser

        win_ds.metadata = md.reset_index(drop=True)

        if hasattr(win_ds, "y"):
            win_ds.y = win_ds.metadata["target"].astype(float).to_numpy()[:, None]

    return windows_concat_ds


# %% ------------------------------------ Utility: keep recordings that have a given event
def keep_only_recordings_with(desc, concat_ds):
    kept = []
    for ds in concat_ds.datasets:
        if np.any(ds.raw.annotations.description == desc):
            kept.append(ds)
        else:
            print(f"[warn] Recording {ds.raw.filenames[0]} does not contain event '{desc}'")
    return BaseConcatDataset(kept)



class SimpleWindowDataset:
    def __init__(self, data_array, metadata):
        """
        data_array: np.array of shape (n_windows, n_channels, n_samples)
        metadata: pd.DataFrame with n_windows rows
        """
        self.X = data_array
        self.metadata = metadata

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        # return a single window + metadata row
        return self.X[idx], self.metadata.iloc[idx]



def create_stimulus_windows_from_baseconcat_with_metadata(
    dataset,
    anchor="stimulus_anchor",
    window_len_s=2.0,
    shift_after_stim_s=0.5,
    sfreq=100,
):
    import numpy as np
    import pandas as pd
    from braindecode.datasets import BaseConcatDataset

    all_windows = []

    window_samples = int(window_len_s * sfreq)
    shift_samples = int(shift_after_stim_s * sfreq)

    for ds in dataset.datasets:
        raw = ds.raw.get_data()  # (n_channels, n_times)
        annotations = ds.raw.annotations
        stim_indices = [i for i, d in enumerate(annotations.description) if d == anchor]
        onsets_samples = (annotations.onset[stim_indices] * sfreq).astype(int)

        data_windows = []
        metadata_list = []

        for i, onset in enumerate(onsets_samples):
            start = onset + shift_samples
            stop = start + window_samples
            if stop > raw.shape[1]:
                continue
            data_windows.append(raw[:, start:stop])
            metadata_list.append({"stim_onset_sample": onset, "i_window_in_trial": i})

        if len(data_windows) == 0:
            continue

        data_array = np.stack(data_windows)
        metadata_df = pd.DataFrame(metadata_list)
        windows_ds = SimpleWindowDataset(data_array, metadata_df)
        all_windows.append(windows_ds)

    return BaseConcatDataset(all_windows)






So, on our raw data, we fit the events present in it, and create a window of interest.

In [70]:
from braindecode.preprocessing import preprocess, Preprocessor, create_windows_from_events
# from eegdash.hbn.windows import (
#     annotate_trials_with_target,
#     add_aux_anchors,
#     add_extras_columns,
#     keep_only_recordings_with,
# )


def trim_recordings_to_trial_multiple(dataset, sfreq, shift_after_stim, window_len):
    """
    Trim each recording in dataset so its length (in samples) is divisible by the trial length.
    
    Parameters
    ----------
    dataset : BaseConcatDataset
        Dataset containing multiple subjects/recordings.
    sfreq : int
        Sampling frequency in Hz.
    shift_after_stim : float
        Time after stimulus onset (in seconds).
    window_len : float
        Duration of the trial window (in seconds).
    """
    trial_start_offset_samples = int(shift_after_stim * sfreq)
    trial_stop_offset_samples = int((shift_after_stim + window_len) * sfreq)
    trial_length_samples = trial_stop_offset_samples - trial_start_offset_samples

    for ds in dataset.datasets:
        n_samples = ds.raw.n_times
        remainder = n_samples % trial_length_samples
        if remainder != 0:
            new_length = n_samples - remainder
            new_tmax = (new_length - 1) / sfreq
            print(f"Trimming {remainder} samples from dataset (old: {n_samples}, new: {new_length})")
            ds.raw.crop(tmax=new_tmax)
    
    return dataset


EPOCH_LEN_S = 2.0
SFREQ = 100 # by definition here

transformation_offline = [
    Preprocessor(
        annotate_trials_with_target,
        target_field="rt_from_stimulus", epoch_length=EPOCH_LEN_S,
        require_stimulus=True, require_response=True,
        apply_on_array=False,
    ),
    Preprocessor(add_aux_anchors, apply_on_array=False),
]

for subject in dataset_ccd:
    preprocess(subject, transformation_offline, n_jobs=1)
# must check if this replaces the original in dataset_ccd
preprocess(test_ccd, transformation_offline, n_jobs=1)



ANCHOR = "stimulus_anchor"

# SHIFT_AFTER_STIM = 0.5
SHIFT_AFTER_STIM = 0.5
WINDOW_LEN       = 2.0


concat_data_array = []
# Keep only recordings that actually contain stimulus anchors
for subject in dataset_ccd:
    concat_data_array.append(keep_only_recordings_with(ANCHOR, subject))
train_dataset = BaseConcatDataset(concat_data_array)
print(train_dataset.__len__())

test_dataset = keep_only_recordings_with(ANCHOR, test_ccd)

# Create single-interval windows (stim-locked, long enough to include the response)

single_windows = create_stimulus_windows_from_baseconcat_with_metadata(
    train_dataset,
    anchor="stimulus_anchor",
    window_len_s=2.0,
    shift_after_stim_s=0.5,
    sfreq=SFREQ,
)

# single_windows = create_windows_from_events(
#     train_dataset,
#     mapping={ANCHOR: 0},
#     trial_start_offset_samples=int(SHIFT_AFTER_STIM * SFREQ),                 # +0.5 s
#     trial_stop_offset_samples=int((SHIFT_AFTER_STIM + WINDOW_LEN) * SFREQ),   # +2.5 s
#     window_size_samples=int(EPOCH_LEN_S * SFREQ),
#     window_stride_samples=SFREQ,
#     drop_last_window=True,
#     drop_bad_windows=True,
#     preload=True,
#     use_mne_epochs=True,
#     on_missing='ignore',
#     accepted_bads_ratio=1.0,
# )

# Injecting metadata into the extra mne annotation.
single_windows = add_extras_columns(
    single_windows,
    train_dataset,
    desc=ANCHOR,
    keys=("target", "rt_from_stimulus", "rt_from_trialstart",
          "stimulus_onset", "response_onset", "correct", "response_type")
          )



#repeat but with validation dataset
valid_single_windows = create_windows_from_events(
    test_dataset,
    mapping={ANCHOR: 0},
    trial_start_offset_samples=int(SHIFT_AFTER_STIM * SFREQ),                 # +0.5 s
    trial_stop_offset_samples=int((SHIFT_AFTER_STIM + WINDOW_LEN) * SFREQ),   # +2.5 s
    window_size_samples=int(EPOCH_LEN_S * SFREQ),
    window_stride_samples=SFREQ,
    preload=True,
    use_mne_epochs=True,
    drop_last_window=True,
    drop_bad_windows=True,
    on_missing='ignore',
    accepted_bads_ratio=1.0,
)

# Injecting metadata into the extra mne annotation.
valid_single_windows = add_extras_columns(
    valid_single_windows,
    test_dataset,
    desc=ANCHOR,
    keys=("target", "rt_from_stimulus", "rt_from_trialstart",
          "stimulus_onset", "response_onset", "correct", "response_type")
          )

C:\Users\benja\AppData\Local\Temp\ipykernel_5064\3798845137.py:128: RuntimeWarning: Omitted 23 annotation(s) that were outside data range.
  raw.set_annotations(new_ann, verbose=False)
C:\Users\benja\AppData\Local\Temp\ipykernel_5064\3798845137.py:128: RuntimeWarning: Omitted 2 annotation(s) that were outside data range.
  raw.set_annotations(new_ann, verbose=False)
C:\Users\benja\AppData\Local\Temp\ipykernel_5064\3798845137.py:175: RuntimeWarning: Omitted 1 annotation(s) that were outside data range.
  raw.set_annotations(ann + aux, verbose=False)


[warn] Recording c:\Users\benja\Documents\VSCode\EEG_Challenge\HBN-EEG\data\ds005515\sub-NDARKM061NHZ\eeg\sub-NDARKM061NHZ_task-contrastChangeDetection_run-2_eeg.bdf does not contain event 'stimulus_anchor'
19636900
Used Annotations descriptions: [np.str_('stimulus_anchor')]
Used Annotations descriptions: [np.str_('stimulus_anchor')]


c:\Users\benja\Documents\VSCode\EEG_Challenge\HBN-EEG\eegchallenge\Lib\site-packages\braindecode\preprocessing\windowers.py:306: UserWarning: Drop bad windows only has an effect if mne epochs are created, and this argument may be removed in the future.
  warnings.warn(


Used Annotations descriptions: [np.str_('stimulus_anchor')]
Used Annotations descriptions: [np.str_('stimulus_anchor')]
Used Annotations descriptions: [np.str_('stimulus_anchor')]
Used Annotations descriptions: [np.str_('stimulus_anchor')]
Used Annotations descriptions: [np.str_('stimulus_anchor')]
Used Annotations descriptions: [np.str_('stimulus_anchor')]
Used Annotations descriptions: [np.str_('stimulus_anchor')]
Used Annotations descriptions: [np.str_('stimulus_anchor')]
Used Annotations descriptions: [np.str_('stimulus_anchor')]
Used Annotations descriptions: [np.str_('stimulus_anchor')]
Used Annotations descriptions: [np.str_('stimulus_anchor')]
Used Annotations descriptions: [np.str_('stimulus_anchor')]
Used Annotations descriptions: [np.str_('stimulus_anchor')]
Used Annotations descriptions: [np.str_('stimulus_anchor')]
Used Annotations descriptions: [np.str_('stimulus_anchor')]
Used Annotations descriptions: [np.str_('stimulus_anchor')]
Used Annotations descriptions: [np.str_(

AttributeError: 'WindowsDataset' object has no attribute 'metadata'

Now that we have our windowed data, we can split it into the different sets that are needed for modeling. Since our challenge focuses on generalization across subjects, we recommend dividing at the subject level.

(1) the training set is used to learn the parameters of our deep learning model,  

(2) the validation set is used to monitor the training process and decide when to stop it, and  

(3) the test set is used to provide an estimate of the generalization performance of our model.

Here, we use the last 10% of windows for testing, 10% for validation and split the remaining 80% of windows into training.

**Here we go into the steps that you and your team must validate to obtain better results**

In [ ]:
# for each windows, we can extract the metainformation using:

train_meta_information = single_windows.get_metadata()

test_meta_information = valid_single_windows.get_metadata()

In [ ]:
train_meta_information.head()
test_meta_information.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils import check_random_state

valid_frac = 0.2
test_frac = 0.01
seed = 2025

train_subjects = train_meta_information["subject"].unique()
test_subjects = test_meta_information["subject"].unique()

train_subj, valid_subj = train_test_split(
    train_subjects, test_size=(valid_frac), random_state=check_random_state(seed), shuffle=True
)
# takes 0.80 of R1-11 for training
# takes 0.20 of R1-11 for validations

# test_subj = train_test_split(test_subjects, test_size=test_frac, random_state=check_random_state(seed + 1), shuffle=True)
test_subj = test_subjects

# takes 1.0 of R5 for test

# sanity check
# assert (set(valid_subj) | set(test_subj) | set(train_subj)) == set(subjects)

In [ ]:
# and finally using braindecode split function, we can do:
subject_split = single_windows.split("subject")
test_split = valid_single_windows.split("subject")

train_set = []
valid_set = []
test_set = []

for s in subject_split:
    if s in train_subj:
        train_set.append(subject_split[s])
    elif s in valid_subj:
        valid_set.append(subject_split[s])
for s in test_split:
    if s in test_subj:
        test_set.append(test_split[s])

train_set = BaseConcatDataset(train_set)
valid_set = BaseConcatDataset(valid_set)
test_set = BaseConcatDataset(test_set)

print("Number of examples in each split in the minirelease")
print(f"Train:\t{len(train_set)}")
print(f"Valid:\t{len(valid_set)}")
print(f"Test:\t{len(test_set)}")
print(test_set)
# print(test_set[0])
# print(test_set[1])
# print(test_set[2])
# print(test_set[3])

Number of examples in each split in the minirelease
Train:	1187
Valid:	244
Test:	82


In [ ]:
# print the testing set std
from numpy import std
test_std= std(np.array([test_set[i][1][0] for i in range(len(test_set))]))
print (test_std)

Finally, we create pytorch `DataLoader`s, which will be used to feed the data to the model during training and evaluation:

In [ ]:
# Create datasets and dataloaders
from torch.utils.data import DataLoader

batch_size = 128
num_workers = 1 # We are using a single worker, but you can increase this for faster data loading

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=num_workers)
valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=num_workers)

## Building the deep learning model

For neural network models, **to start**, we suggest using [braindecode models](https://braindecode.org/1.2/models/models_table.html) zoo. We have implemented several different models for decoding the brain timeseries.

Your team's responsibility is to develop a PyTorch module that receives the three-dimensional (`batch`, `n_chans`, `n_times`) input and outputs the contrastive response time.

In [ ]:
# from braindecode.models.util import models_dict

# names = sorted(models_dict)
# w = max(len(n) for n in names)

# for i in range(0, len(names), 3):
#     row = names[i:i+3]
#     print("  ".join(f"{n:<{w}}" for n in row))
    
# ATCNet                  AttentionBaseNet        AttnSleep             
# BDTCN                   BIOT                    CTNet                 
# ContraWR                Deep4Net                DeepSleepNet          
# EEGConformer            EEGITNet                EEGInceptionERP       
# EEGInceptionMI          EEGMiner                EEGNeX                
# EEGNet                  EEGSimpleConv           EEGTCNet              
# FBCNet                  FBLightConvNet          FBMSNet               
# IFNet                   Labram                  MSVTNet               
# SCCNet                  SPARCNet                ShallowFBCSPNet       
# SignalJEPA              SignalJEPA_Contextual   SignalJEPA_PostLocal  
# SignalJEPA_PreLocal     SincShallowNet          SleepStagerBlanco2020 
# SleepStagerChambon2018  SyncNet                 TIDNet                
# TSception               USleep                

ATCNet                  AttentionBaseNet        AttnSleep             
BDTCN                   BIOT                    CTNet                 
ContraWR                Deep4Net                DeepSleepNet          
EEGConformer            EEGITNet                EEGInceptionERP       
EEGInceptionMI          EEGMiner                EEGNeX                
EEGNet                  EEGSimpleConv           EEGTCNet              
FBCNet                  FBLightConvNet          FBMSNet               
IFNet                   Labram                  MSVTNet               
SCCNet                  SPARCNet                ShallowFBCSPNet       
SignalJEPA              SignalJEPA_Contextual   SignalJEPA_PostLocal  
SignalJEPA_PreLocal     SincShallowNet          SleepStagerBlanco2020 
SleepStagerChambon2018  SyncNet                 TIDNet                
TSception               USleep                


In [ ]:
# for any braindecode model, you can initialize only inputing the signal related parameters
from braindecode import EEGSimpleConv

# ModelClass = getattr(models, input)
# print (input)
# model = ModelClass


model = EEGSimpleConv(n_chans=129, # 129 channels
                n_outputs=1, # 1 output for regression
                n_times=200, #2 seconds
                sfreq=100,      # sample frequency 100 Hz
                )

### The rest is our classic PyTorch/torch lighting/skorch training pipeline

In [ ]:
# Defining training parameters

lr = 1E-3
weight_decay = 1E-5
n_epochs = 100
early_stopping_patience = 50

In [ ]:
from typing import Optional
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from torch.nn import Module
from torch.optim.lr_scheduler import LRScheduler

# Define a method for training one epoch
def train_one_epoch(
    dataloader: DataLoader,
    model: Module,
    loss_fn,
    optimizer,
    scheduler: Optional[LRScheduler],
    epoch: int,
    device,
    print_batch_stats: bool = True,
):
    model.train()

    total_loss = 0.0
    sum_sq_err = 0.0
    n_samples = 0

    progress_bar = tqdm(
        enumerate(dataloader), total=len(dataloader), disable=not print_batch_stats
    )

    for batch_idx, batch in progress_bar:
        # Support datasets that may return (X, y) or (X, y, ...)
        X, y = batch[0], batch[1]
        X, y = X.to(device).float(), y.to(device).float()

        optimizer.zero_grad(set_to_none=True)
        preds = model(X)
        loss = loss_fn(preds, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Flatten to 1D for regression metrics and accumulate squared error
        preds_flat = preds.detach().view(-1)
        y_flat = y.detach().view(-1)
        sum_sq_err += torch.sum((preds_flat - y_flat) ** 2).item()
        n_samples += y_flat.numel()

        if print_batch_stats:
            running_rmse = (sum_sq_err / max(n_samples, 1)) ** 0.5
            progress_bar.set_description(
                f"Epoch {epoch}, Batch {batch_idx + 1}/{len(dataloader)}, "
                f"Loss: {loss.item():.6f}, RMSE: {running_rmse:.6f}"
            )

    if scheduler is not None:
        scheduler.step()

    avg_loss = total_loss / len(dataloader)
    rmse = (sum_sq_err / max(n_samples, 1)) ** 0.5
    return avg_loss, rmse

In [ ]:
import torch
from torch.utils.data import DataLoader
from torch.nn import Module
from tqdm import tqdm


@torch.no_grad()
def valid_model(
    dataloader: DataLoader,
    model: Module,
    loss_fn,
    device,
    print_batch_stats: bool = True,
):
    model.eval()

    total_loss = 0.0
    sum_sq_err = 0.0
    n_batches = len(dataloader)
    n_samples = 0

    iterator = tqdm(
        enumerate(dataloader),
        total=n_batches,
        disable=not print_batch_stats
    )
    
    all_y_trues = []

    for batch_idx, batch in iterator:
        # Supports (X, y) or (X, y, ...)
        X, y = batch[0], batch[1]
        X, y = X.to(device).float(), y.to(device).float()
        # casting X to float32

        preds = model(X)
        batch_loss = loss_fn(preds, y).item()
        total_loss += batch_loss

        preds_flat = preds.detach().view(-1)
        y_flat = y.detach().view(-1)
        sum_sq_err += torch.sum((preds_flat - y_flat) ** 2).item()
        n_samples += y_flat.numel()
        
        all_y_trues.append(y_flat.cpu())  # Chen: added for score

        if print_batch_stats:
            running_rmse = (sum_sq_err / max(n_samples, 1)) ** 0.5
            iterator.set_description(
                f"Val Batch {batch_idx + 1}/{n_batches}, "
                f"Loss: {batch_loss:.6f}, RMSE: {running_rmse:.6f}"
            )

    avg_loss = total_loss / n_batches if n_batches else float("nan")
    rmse = (sum_sq_err / max(n_samples, 1)) ** 0.5

    from numpy import std # Chen: added for score
    all_y_trues = torch.cat(all_y_trues).numpy() # Chen: added for score
    std_gt =std(all_y_trues) if std(all_y_trues) != 0 else float("nan") # Chen: added for score
    score = rmse / std(all_y_trues) if std(all_y_trues) != 0 else float("nan") # Chen: added for score
    # print(f"Val RMSE: {rmse:.6f}, Val Loss: {avg_loss:.6f}\n")
    print(f"Val RMSE: {rmse:.6f}, Val Loss: {avg_loss:.6f}, score: {score:.6f}stdytrue{std_gt}\n")
    return avg_loss, rmse,score,std(all_y_trues)


In [ ]:
import copy

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs - 1)
loss_fn = torch.nn.MSELoss()

patience = 5
min_delta = 1e-4
best_rmse = float("inf")
epochs_no_improve = 0
best_state, best_epoch = None, None

for epoch in range(1, n_epochs + 1):
    print(f"Epoch {epoch}/{n_epochs}: ", end="")

    train_loss, train_rmse = train_one_epoch(
        train_loader, model, loss_fn, optimizer, scheduler, epoch, device
    )
    val_loss, val_rmse,score,std_gt = valid_model(test_loader, model, loss_fn, device)

    print(
        f"Train RMSE: {train_rmse:.6f}, "
        f"Average Train Loss: {train_loss:.6f}, "
        f"Val RMSE: {val_rmse:.6f}, "
        f"Average Val Loss: {val_loss:.6f} "
        f"Score: {score:.6f} "
        f"std_gt: {std_gt:.6f}"
    )

    if val_rmse < best_rmse - min_delta:
        best_rmse = val_rmse
        best_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch}. Best Val RMSE: {best_rmse:.6f} (epoch {best_epoch})")
            break

if best_state is not None:
    model.load_state_dict(best_state)


In [ ]:
# saving the model

torch.save(model.state_dict(), "model.pth")


Here are some of the available datasets in `braindecode`. For a comprehensive list, please refer to the official `braindecode` documentation.

- **BCI-IV 2a**: Dataset for motor imagery.
- **BCI-IV 2b**: Dataset for motor imagery.
- **PhysioBank MIMIC III**: Large dataset with various paradigms.
- **Sleep Physionet**: Dataset for sleep stage classification.